# Piece 2 — joint sentence + token head

`loss = token_loss + lambda * sentence_loss`, sweeping lambda over
0, 0.1, 0.3, 0.5, 1.0.

**Lambda 0 is the control.** At lambda 0 the script builds the model with no
sentence head at all, so that row is the Piece 1 architecture exactly and the
experiment stands on its own — no waiting for anyone else.

Everything is chosen on validation. Test is never scored here.

### Setup
1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Set `OWNER` in the Drive cell
3. Run top to bottom (~2.5 h)

## 1. GPU

In [ ]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'

## 2. Install

In [ ]:
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('ready')

## 3. Code

In [ ]:
REPO = 'https://github.com/hatheem-r/project_DNN.git'

import os, shutil
if os.path.exists('/content/project'): shutil.rmtree('/content/project')
%cd /content
!git clone -q $REPO project
%cd /content/project
!ls src/ notebooks/

## 4. Safety tests — do not skip

The alignment tests catch a bug that would silently shift labels against words
while the loss still falls and nothing crashes.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## 5. Drive, so a disconnect does not lose the run

Colab wipes the session when it ends or times out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sold_results'
import os
os.makedirs(OUT, exist_ok=True)

OWNER = 'YOUR_NAME'        # <-- EDIT before running: recorded in every results row

print('saving to', OUT, '| owner', OWNER)

## 6. fastText vectors

About 460 MB.

In [ ]:
!mkdir -p embeddings results artifacts
![ -f embeddings/cc.si.300.vec.gz ] || wget -q --show-progress -O embeddings/cc.si.300.vec.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
!ls -lh embeddings/

## 7. Smoke test — 1 seed, 3 epochs

Runs the Piece 2 path so the sentence head is exercised before committing hours
to it. The score will be poor; we only care that it does not crash.

The smoke run appends junk rows to `results/results_piece2.csv`, so the second
line deletes that file and the real run starts clean.

In [ ]:
!python notebooks/09_pieces_234.py --piece2 --seeds 1 --epochs 3 --patience 2 --owner $OWNER 2>&1 | tail -20
!rm -f results/results_piece2.csv && echo '--- smoke rows cleared, ready for the real run ---'

## 8. The run

5 seeds, not the default 3, because the project rule is that every reported
number is a mean over 5 seeds with its standard deviation. ~2.5 h on a T4.

`--loss` is left at its default (`cross_entropy`).

**Read precision and recall, not just F1.** After Piece 1 the model sits near
P 0.74 / R 0.68, close to balanced, so there is little headroom. A lambda that
lifts recall while collapsing precision is a net loss.

**Expected: the lambda 0 row lands near 0.704**, not the 0.7083 quoted for
Piece 1. Piece 1's headline used CRF on / batch 32 / 5 seeds; this runs CRF off
/ batch 64. Same architecture, different training config. Compare every lambda
against this run's own lambda 0 row.

A null is the likely outcome. Report the best lambda regardless — Piece 4
cannot run without a sentence head, because SemiSOLD's teacher scores are
sentence level and distillation has nothing to attach to otherwise.

In [ ]:
!python notebooks/09_pieces_234.py --piece2 --owner $OWNER --seeds 1 2 3 4 5 2>&1 | tee results/piece2_report.txt | tail -40
!cp results/results_piece2.csv results/piece2_report.txt $OUT/ && ls -la $OUT/

## 9. Report to the group

- lambda = 0 (control): F1 ± std
- best lambda, its F1 ± std, and its P / R
- whether it beat lambda = 0 (the script prints the verdict)
- the lambda value Piece 4 should use

Then commit:

```
git add results/results_piece2.csv results/piece2_report.txt
git commit -m "piece2 results"
git pull --rebase
git push
```